# Day 5 lab: Shuffle and joins: how Spark picks a join

**Goal:** Explain what a shuffle costs and how each join algorithm works, predict which one Spark will pick, and change its choice on purpose.

**Time:** about 2 h 30 min. Checklist: 3.1–3.10, 3.12 · 🧭 join selection.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

In [ ]:
from lab_utils import *

spark = get_spark()

## 📘 Code along

Type the code yourself instead of pasting it, run every cell, and keep the Spark UI open. Swap in `spark = get_spark()` from the starter kit wherever an example builds its own session.

### Task 1. Joins in the official guides — 35 min

Type the join examples from three official pages: the last section of Chapter 3, "When DataFrames Collide" (inner, left and right joins on `name`); every join type on the SQL reference JOIN page (inner, left, right, full, cross, semi, anti); and the Examples of `DataFrame.join` in the API reference (joining on a column name, on expressions, on several keys). Then type the `DataFrame.hint` examples. Run `explain()` as you go and note which joins show BroadcastHashJoin and which SortMergeJoin.

[Joins in the official guides](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html) · [Chapter 3 joins](https://spark.apache.org/docs/latest/api/python/user_guide/dataprep.html) · [DataFrame.join API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html) · [DataFrame.hint API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.hint.html)

In [ ]:
# Task 1: your code here


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

## ✍️ Exercises

Write these from scratch in the cells below, using the patterns you just typed. Each one ends with a Check you can verify yourself; open the reference solution only after you have a working answer.

### Task 2. Exercise 1: every join type, and which side can be broadcast — 30 min

With the chapter's `df1` (4 people) and `df2` (5 people with incomes), write down the row count of inner, left, right, full, left_semi, left_anti and cross first, then check. With AQE off, every join is a SortMergeJoin, even on 4 rows: DataFrames built from Python rows have no size estimate (`df2._jdf.queryExecution().optimizedPlan().stats().sizeInBytes()` prints 9223372036854775807), so Spark treats them as huge. Add `F.broadcast(df2)`, then `F.broadcast(df1)`, to every join and record which hints Spark honours. 

**Check:** 3, 4, 5, 6, 3, 1 and 20 rows; left, semi and anti joins can broadcast only the right side, right joins only the left side, full outer neither. Proves 3.2, 3.7, 3.9.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day05.py)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 3. Exercise 2: predict, then verify the strategy matrix — 35 min

AQE off. Write down your predicted strategy and Exchange count for each case, then check with `count_nodes(df, "BroadcastHashJoin", "ShuffledHashJoin", "SortMergeJoin", "BroadcastNestedLoopJoin", "CartesianProduct", "Exchange")`. With `orders` = `data/orders`, `cust` = `data/customers`, `big2` = `make_orders(spark, 1_000_000, seed=1)`: (a) `orders.join(cust, "customer_id")`; (b) `orders.join(big2, "customer_id")` (explain only); (c) `cust.join(orders, "customer_id", "left")`; (d) `orders.join(cust, "customer_id", "full")`; (e) `orders.join(cust, orders.amount < cust.customer_id)` (explain only); (f) case (e) with `autoBroadcastJoinThreshold` -1; (g) case (b) with `spark.sql.join.preferSortMergeJoin` false and threshold `1MB`, joining a 200,000-row `make_orders(spark, 200_000, seed=2)`. Expect: broadcast hash; sort-merge; sort-merge (the small left side of a left join cannot be the build side); sort-merge; broadcast nested loop; Cartesian product; shuffled hash. Proves 3.9, 3.2, 3.8.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day05.py)

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

## 🔬 Prove it

Short, focused checks of today's checklist questions in the plan and the Spark UI.

### Task 4. AQE switches a join at runtime — 25 min

Write `make_orders(spark, 1_000_000, seed=3)` to `data/orders_b`. `ob = spark.read.parquet("data/orders_b").filter("amount < 0.1")` keeps about 1,000 rows. Join orders with `ob` on customer_id, then `groupBy(orders.status).agg(F.count(ob.note))`; counting `ob.note` keeps a wide column on that side, so its static estimate is about 20 MB (Spark scales file size by the columns a query reads, and ignores the filter). AQE off: SortMergeJoin. AQE on: build it again, `collect()` it, then `explain()`: the final plan shows BroadcastHashJoin and `AQEShuffleRead local`. Count only `ob.customer_id` instead and even the static plan broadcasts: pruning alone pushes the estimate under 10 MB. Proves 3.10, 3.2.

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 5. A join with no shuffle: bucketing — 25 min

Write orders and `big2` with `bucketBy(16, "customer_id").sortBy("customer_id").saveAsTable(...)`; `DESCRIBE EXTENDED` shows the bucket spec. With broadcasting off, join the two tables: no Exchange under the SortMergeJoin (the Sort stays, as expected since Spark 3). Re-bucket one side with 8 buckets: one side shuffles again; set `spark.sql.bucketing.coalesceBucketsInJoin.enabled` to true and it goes away. Join on a column that is not the bucket column: both Exchanges return. Proves 3.5, 3.4.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*